In [ ]:
import pymysql
import pandas as pd

# 建立连接（你可以替换自己的连接信息）
conn = pymysql.connect(
    host='###',
    port=3306,
    user='###',
    password='###',
    database='ch00',
    charset='utf8'
)

query_tickers = """
SELECT DISTINCT ticker
FROM pplog
WHERE boid = 43
"""

df_tickers = pd.read_sql(query_tickers, conn)
tickers = df_tickers['ticker'].tolist()

# 构造 ticker 列表为 SQL IN (...) 语法
tickers_str = ",".join(f"'{t}'" for t in tickers)

query_prices = f"""
SELECT dt, ticker, close
FROM pplog
WHERE ticker IN ({tickers_str})
  AND dt BETWEEN '2025-01-01' AND '2025-12-31'
ORDER BY dt, ticker
"""

df_prices = pd.read_sql(query_prices, conn)
df_prices = df_prices.drop_duplicates(subset=['dt', 'ticker'])

# 转换为日期格式
df_prices['dt'] = pd.to_datetime(df_prices['dt'])


# 转为行：日期，列：ticker，值：close
df_close = df_prices.pivot(index='dt', columns='ticker', values='close')

query_full = """
SELECT dt, ticker, share, close, ecls
FROM ppLog
WHERE boid = 43
  AND dt BETWEEN '2025-01-01' AND '2025-12-31'
ORDER BY dt, ticker
"""
df_full = pd.read_sql(query_full, conn)
print(df_full)

In [ ]:
# Step 1: 计算每一行股票的收益率（基于实际持仓）
df_full['return'] = (df_full['close'] - df_full['ecls']) / df_full['ecls']

# 强制转换为浮点数
df_full['share'] = pd.to_numeric(df_full['share'], errors='coerce')
df_full['close'] = pd.to_numeric(df_full['close'], errors='coerce')
# Step 2: 计算市值（权重计算用）
df_full['value'] = df_full['share'] * df_full['close']


# 计算总市值（用绝对值）
df_full['abs_value'] = df_full['value'].abs()

# 每日总市值（绝对值之和）
daily_abs_value = df_full.groupby('dt')['abs_value'].sum().rename('total_value')

# 合并 total_value（对齐 dt）
df_full = df_full.merge(daily_abs_value, on='dt')

# 保留做空方向的权重计算
df_full['weight'] = df_full['value'] / df_full['total_value']
print(df_full)

In [ ]:
# Step 1：每只股票的加权收益
df_full['weighted_return'] = df_full['return'] * df_full['weight']

# Step 2：每天组合收益率 = 加权收益之和
portfolio_returns = df_full.groupby('dt')['weighted_return'].sum().sort_index()

print(portfolio_returns)

cumulative_return = (1 + portfolio_returns).cumprod()

print(cumulative_return)

target_date = pd.to_datetime('2025-05-12').date()
result = cumulative_return.get(target_date)
print(result)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(cumulative_return.index, cumulative_return.values, label='组合累计收益')
plt.title("动态持仓组合累计收益率")
plt.xlabel("日期")
plt.ylabel("累计收益")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_positive = df_full[df_full['value'] > 0].copy()
print(df_positive)
positive_daily_value = df_positive.groupby('dt')['value'].sum().rename('total_value_1')
df_positive = df_positive.merge(positive_daily_value, on='dt')
df_positive['weight_1'] = df_positive['value'] / df_positive['total_value_1']
df_positive['weighted_return_1'] = df_positive['weight_1'] * df_positive['return']
portfolio_return_pos = df_positive.groupby('dt')['weighted_return_1'].sum().sort_index()
cumulative_return_pos = (1 + portfolio_return_pos).cumprod()
print(cumulative_return_pos)

plt.figure(figsize=(12,6))
plt.plot(cumulative_return_pos.index, cumulative_return_pos.values, label='组合累计收益')
plt.title("动态持仓组合累计收益率")
plt.xlabel("日期")
plt.ylabel("累计收益")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()